# E49 --- a volta e um produto

**A tentativa.** A proposicao do capitulo assina a metade contida da volta: com passo
deterministico o ganho e exatamente beta, e a profundidade se compra em beta elevado ao horizonte.
O que a proposicao nao toca e a outra metade --- o passo de cada dia SORTEIA. Aqui a volta e lida
como o produto dos ganhos aleatorios, antes de o desenho medir.

**O que se mede.**

1. a influencia do dia zero (o produto acumulado) contra o horizonte, em duas calibracoes com o
   MESMO envelope --- a do capitulo e uma mais agitada;
2. as duas contas da mesma soltura: a mediana das trilhas, que desce no expoente medido, e a media,
   que desce na conta log(alfa+beta);
3. a fracao de mundos que cresce antes de cair, e a fracao de janelas de espera cujo produto bate a
   conta da media.

**Convencoes** (AGENTS.md paragrafos 7 e 9): um experimento por caderno, parametros no topo
marcados com "brinque com", algoritmo em frevolab, resultado em lab/resultados/E49_produto.json,
figuras em .pdf e .png.

In [1]:
# <- brinque com: MUNDOS, DIAS, HORIZONTES, ESPERA, ALFA_CAPITULO, BETA_CAPITULO, ALFA_CONTRASTE, BETA_CONTRASTE, SEMENTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

import frevolab
from frevolab import graficos, intervencao

MUNDOS = 4000                  # os mundos do mesmo sorteio (uma linha por mundo)
DIAS = 252                     # o horizonte da contencao do capitulo
HORIZONTES = (1, 2, 5, 10, 20, 40, 80, 160, 252)
ESPERA = 20                    # a janela que conta as fracoes
ALFA_CAPITULO, BETA_CAPITULO = 0.02, 0.97      # a calibracao do capitulo
ALFA_CONTRASTE, BETA_CONTRASTE = 0.08, 0.91    # o contraste de MESMO envelope (0,99)
SEMENTE = 116                  # E40..E48 usam 107..115

print("frevolab %s | %d mundos de %d dias | o sorteio do produto | semente %d"
      % (frevolab.VERSAO, MUNDOS, DIAS, SEMENTE))

frevolab 0.1.0 | 4000 mundos de 252 dias | o sorteio do produto | semente 116


## As duas contas da mesma soltura

A mediana e a media das trilhas nao descem juntas: uma desce no expoente (a media dos log-ganhos),
a outra na conta da media. A distancia entre as duas e o preco de ler o produto pela media.

In [2]:
rng = np.random.default_rng(SEMENTE)
trilhas = {}
for nome, (alfa, beta) in (("capitulo", (ALFA_CAPITULO, BETA_CAPITULO)),
                           ("contraste", (ALFA_CONTRASTE, BETA_CONTRASTE))):
    g = intervencao.ganhos((MUNDOS, DIAS), rng, alfa=alfa, beta=beta)
    trilhas[nome] = intervencao.resumo_do_produto(g, HORIZONTES, ESPERA)
    r = trilhas[nome]
    print("%-10s expoente %+.5f | conta da media %+.5f | vao %+.5f"
          % (nome, r["expoente"], r["conta"], r["expoente"] - r["conta"]))
    print("%-10s cresce antes de cair %.4f | janelas acima da conta %.4f"
          % (nome, r["cresce_antes_de_cair"], r["janelas_acima_da_conta"]))

capitulo   expoente -0.01045 | conta da media -0.01006 | vao -0.00039
capitulo   cresce antes de cair 0.4890 | janelas acima da conta 0.4365
contraste  expoente -0.01554 | conta da media -0.00999 | vao -0.00555
contraste  cresce antes de cair 0.7305 | janelas acima da conta 0.3740


In [3]:
# Figura 1: a influencia do dia zero contra o horizonte, nas duas calibracoes.
fig, eixos = plt.subplots(1, 2, figsize=(10.0, 3.9), sharey=True)
for eixo, nome in zip(eixos, ("capitulo", "contraste")):
    r = trilhas[nome]
    hs = np.array(r["horizontes"])
    eixo.plot(hs, r["mediana"], "o-", color="#1f4e79", ms=4, label="a mediana dos mundos")
    eixo.plot(hs, r["media"], "s-", color="#c78f2c", ms=4, label="a media dos mundos")
    alfa, beta = ((ALFA_CAPITULO, BETA_CAPITULO) if nome == "capitulo"
                  else (ALFA_CONTRASTE, BETA_CONTRASTE))
    eixo.plot(hs, beta ** hs, ":", color="#555555", lw=1.4, label="beta elevado ao horizonte")
    eixo.set_yscale("log")
    eixo.set_xscale("log")
    eixo.set_title("%s (envelope %.2f)" % (nome, alfa + beta), fontsize=10)
    eixo.set_xlabel("horizonte (dias)")
    eixo.legend(fontsize=7)
eixos[0].set_ylabel("influencia do dia zero")
fig.suptitle("a volta e um produto: mediana, media e o chao do ganho", fontsize=10)
fig.tight_layout()
graficos.salvar(fig, "E49_produto", 1)
plt.close(fig)
print("figura E49_produto_1 salva")

figura E49_produto_1 salva


In [4]:
# Figura 2: os desviantes --- quem cresce antes de cair, e quem bate a conta da media.
fig, eixos = plt.subplots(1, 2, figsize=(9.6, 3.7))
nomes = ["capitulo", "contraste"]
eixos[0].bar(nomes, [100 * trilhas[n]["cresce_antes_de_cair"] for n in nomes], color="#1f4e79")
eixos[0].set_ylabel("mundos que crescem antes de cair (%)")
eixos[0].set_title("a influencia que sobe antes de descer", fontsize=10)
eixos[1].bar(nomes, [100 * trilhas[n]["janelas_acima_da_conta"] for n in nomes], color="#c78f2c")
eixos[1].set_ylabel("janelas de espera acima da conta (%)")
eixos[1].set_title("o produto que bate a conta da media", fontsize=10)
fig.tight_layout()
graficos.salvar(fig, "E49_produto", 2)
plt.close(fig)
print("figura E49_produto_2 salva")

figura E49_produto_2 salva


## Leitura visual das figuras

**Declarada contra os .png depois da execucao** (AGENTS.md paragrafo 9): o criterio de frescor e o
hash das celulas de codigo.

O que as legendas do capitulo afirmam, e a leitura tem de conferir nos .png:

1. **Figura 1**: nos dois paineis, a curva da mediana abaixo da curva da media e as duas acima da
   linha pontilhada do ganho minimo; no contraste as duas se abrem mais.
2. **Figura 2**: as duas barras de cada painel, com as fracoes do contraste maiores que as do
   capitulo.

In [5]:
# O resultado: um objeto por grandeza, em portugues, para o livro citar por comando.
cap = trilhas["capitulo"]
con = trilhas["contraste"]
i_espera = cap["horizontes"].index(ESPERA)
resultado = {
    "produto_mundos": MUNDOS,
    "produto_dias": DIAS,
    "produto_horizontes": len(HORIZONTES),
    "produto_espera": ESPERA,
    "produto_expoente_capitulo": round(cap["expoente"], 5),
    "produto_conta_capitulo": round(cap["conta"], 5),
    "produto_vao_capitulo": round(cap["expoente"] - cap["conta"], 5),
    "produto_expoente_contraste": round(con["expoente"], 5),
    "produto_conta_contraste": round(con["conta"], 5),
    "produto_vao_contraste": round(con["expoente"] - con["conta"], 5),
    "produto_mediana_espera_capitulo": round(cap["mediana"][i_espera], 5),
    "produto_media_espera_capitulo": round(cap["media"][i_espera], 5),
    "produto_mediana_espera_contraste": round(con["mediana"][i_espera], 5),
    "produto_media_espera_contraste": round(con["media"][i_espera], 5),
    "produto_chao_capitulo": round(BETA_CAPITULO ** ESPERA, 5),
    "produto_chao_contraste": round(BETA_CONTRASTE ** ESPERA, 5),
    "produto_cresce_capitulo_pct": round(100 * cap["cresce_antes_de_cair"], 3),
    "produto_cresce_contraste_pct": round(100 * con["cresce_antes_de_cair"], 3),
    "produto_janelas_acima_capitulo_pct": round(100 * cap["janelas_acima_da_conta"], 3),
    "produto_janelas_acima_contraste_pct": round(100 * con["janelas_acima_da_conta"], 3),
}
caminho = Path("lab/resultados/E49_produto.json")
caminho.parent.mkdir(parents=True, exist_ok=True)
caminho.write_text(json.dumps(resultado, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=1))

{
 "produto_mundos": 4000,
 "produto_dias": 252,
 "produto_horizontes": 9,
 "produto_espera": 20,
 "produto_expoente_capitulo": -0.01045,
 "produto_conta_capitulo": -0.01006,
 "produto_vao_capitulo": -0.00039,
 "produto_expoente_contraste": -0.01554,
 "produto_conta_contraste": -0.00999,
 "produto_vao_contraste": -0.00555,
 "produto_mediana_espera_capitulo": 0.79951,
 "produto_media_espera_capitulo": 0.81589,
 "produto_mediana_espera_contraste": 0.70041,
 "produto_media_espera_contraste": 0.81748,
 "produto_chao_capitulo": 0.54379,
 "produto_chao_contraste": 0.15164,
 "produto_cresce_capitulo_pct": 48.9,
 "produto_cresce_contraste_pct": 73.05,
 "produto_janelas_acima_capitulo_pct": 43.65,
 "produto_janelas_acima_contraste_pct": 37.398
}
